# Mock Interview

A randomized, timed, cross-chapter mock interview drawn from `question_bank.json` (93
entries across all 9 chapters). This notebook never loads or displays a model answer —
review your answers against `solutions/question_bank_answers.json` afterward, on your own.

Run this the same way you'd rehearse for a real interview: don't look ahead, actually type
an answer before revealing the follow-up, and don't open the solutions file until the
session is over.

## Setup

In [1]:
import json
import random
import sys
import time
from pathlib import Path

from IPython.core.error import StdinNotImplementedError

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

QUESTION_BANK_PATH = _repo_root / "interview_prep" / "question_bank.json"

with open(QUESTION_BANK_PATH) as f:
    question_bank = json.load(f)

# key_concepts lives with the answers in solutions/, not alongside the questions -- opening
# question_bank.json should not hand you the answer skeleton for all 93 of them. It is loaded
# here only to drive the post-answer keyword signal below, which you never see until after
# you have committed to an answer.
ANSWER_KEY_PATH = _repo_root / "solutions" / "question_bank_answers.json"

with open(ANSWER_KEY_PATH) as f:
    answer_key = json.load(f)


def key_concepts_for(question_id: str) -> list:
    return answer_key.get(question_id, {}).get("key_concepts", [])

print(f"Loaded {len(question_bank)} questions across {len({q['chapter'] for q in question_bank})} chapters.")

Loaded 93 questions across 9 chapters.


In [2]:
def get_answer(prompt: str) -> str:
    '''Wraps input() so this notebook still executes cleanly with zero errors in a headless
    test runner (pytest --nbmake, CI) where there is no real stdin to read from. Plain Python
    with closed stdin raises EOFError; the ZMQ-based Jupyter kernel used by nbconvert/nbmake
    raises IPython's StdinNotImplementedError instead, since the connected frontend does not
    implement stdin requests at all -- both are caught here so this genuinely runs cleanly
    under either execution path. Interactively, this behaves like a normal input() prompt.'''
    print(prompt)
    try:
        return input("Your answer: ")
    except (EOFError, StdinNotImplementedError):
        print("(no interactive input available in this run -- using a placeholder answer "
              "so the rest of this cell logic still executes for real)")
        return "(placeholder answer -- no interactive input available)"


## Sampling questions

In [3]:
def sample_questions(bank: list, n: int = 5, chapter: int = None, difficulty: str = None, seed: int = None) -> list:
    pool = bank
    if chapter is not None:
        pool = [q for q in pool if q["chapter"] == chapter]
    if difficulty is not None:
        pool = [q for q in pool if q["difficulty"] == difficulty]
    rng = random.Random(seed)
    return rng.sample(pool, min(n, len(pool)))


def keyword_feedback(answer_text: str, key_concepts: list) -> str:
    '''A rough, directional signal only -- not a grade. Simple case-insensitive substring
    presence, nothing more sophisticated.'''
    answer_lower = answer_text.lower()
    mentioned = [c for c in key_concepts if c.lower() in answer_lower]
    missed = [c for c in key_concepts if c.lower() not in answer_lower]
    lines = ["(rough keyword signal, not a grade -- see the solutions file for what actually matters)"]
    if mentioned:
        lines.append(f"  you mentioned: {', '.join(mentioned)}")
    if missed:
        lines.append(f"  you didn't mention: {', '.join(missed)}")
    return "\n".join(lines)


## A real branching example

One entry in the question bank (`cost-001`, the GPU-cost seed question) gets a genuinely
different follow-up depending on what the candidate's answer contains — real conditional
logic on keyword matches, not a scripted, static Q&A pair. Demonstrated below with two
canned example answers so the branching itself is visibly exercised, both directions, in
this notebook's own output.

In [4]:
def branching_follow_up(question_id: str, answer_text: str, default_follow_up: str) -> str:
    '''Real conditional branching for the cost-001 seed question specifically -- the kind of
    thing a real interviewer does naturally: react to what you actually said, not read the
    next line off a script.'''
    if question_id != "cost-001":
        return default_follow_up

    answer_lower = answer_text.lower()
    if "more gpu" in answer_lower or "add gpu" in answer_lower or "buy more" in answer_lower:
        return ("Pushback: are you sure more hardware is the fastest lever here? What would "
                 "you check in the request logs *before* spending money on capacity?")
    if "profile" in answer_lower or "log" in answer_lower or "duplicate" in answer_lower or "trace" in answer_lower:
        return ("Good instinct. Say the logs show a token-volume step change on a slice of "
                 "requests with no corresponding change in query complexity -- what's your "
                 "next move?")
    return default_follow_up


cost_question = next(q for q in question_bank if q["id"] == "cost-001")

print("Candidate answer A: \"I'd just add more GPUs to handle the load.\"")
print("Interviewer follow-up:")
print(f"  {branching_follow_up('cost-001', 'I would just add more GPUs to handle the load.', cost_question['follow_up'])}")

print("\nCandidate answer B: \"I'd profile the request logs first to see where the cost is actually coming from.\"")
print("Interviewer follow-up:")
print(f"  {branching_follow_up('cost-001', 'I would profile the request logs first.', cost_question['follow_up'])}")

print("\nCandidate answer C: \"Not sure, maybe check the model version?\"")
print("Interviewer follow-up (neither branch matches -- falls back to the standard question):")
print(f"  {branching_follow_up('cost-001', 'Not sure, maybe check the model version.', cost_question['follow_up'])}")


Candidate answer A: "I'd just add more GPUs to handle the load."
Interviewer follow-up:
  Pushback: are you sure more hardware is the fastest lever here? What would you check in the request logs *before* spending money on capacity?

Candidate answer B: "I'd profile the request logs first to see where the cost is actually coming from."
Interviewer follow-up:
  Good instinct. Say the logs show a token-volume step change on a slice of requests with no corresponding change in query complexity -- what's your next move?

Candidate answer C: "Not sure, maybe check the model version?"
Interviewer follow-up (neither branch matches -- falls back to the standard question):
  What optimizations would you try before adding more GPUs?


## Run a session

Configure below: how many questions, and optionally filter by chapter (1-9) or difficulty
(`"junior"`, `"mid"`, `"senior"`). Leave `CHAPTER`/`DIFFICULTY` as `None` for a real
cross-chapter mock interview.

In [5]:
N_QUESTIONS = 5
CHAPTER = None       # e.g. 6 to drill only Chapter 6
DIFFICULTY = None    # e.g. "senior"
SESSION_SEED = None  # set an int for a reproducible session; None for a different sample each run

session_questions = sample_questions(question_bank, n=N_QUESTIONS, chapter=CHAPTER, difficulty=DIFFICULTY, seed=SESSION_SEED)
print(f"Session ready: {len(session_questions)} questions.")


Session ready: 5 questions.


In [6]:
def run_session(questions: list) -> list:
    asked_ids = []
    for i, q in enumerate(questions, start=1):
        print(f"\n{'=' * 70}")
        print(f"Question {i}/{len(questions)}  --  Chapter {q['chapter']}, {q['difficulty']}, {q['category']}")
        print("=" * 70)

        start = time.time()
        answer = get_answer(q["scenario"])
        elapsed = time.time() - start
        print(f"\n(elapsed: {elapsed:.0f}s -- soft timer, nothing was cut off)")

        print(keyword_feedback(answer, key_concepts_for(q["id"])))

        follow_up = branching_follow_up(q["id"], answer, q["follow_up"])
        print(f"\nFollow-up: {follow_up}")

        asked_ids.append(q["id"])
    return asked_ids


asked = run_session(session_questions)



Question 1/5  --  Chapter 4, mid, Production reliability
You shorten a cache's TTL from one hour to five minutes to reduce staleness complaints.
(no interactive input available in this run -- using a placeholder answer so the rest of this cell logic still executes for real)

(elapsed: 0s -- soft timer, nothing was cut off)
(rough keyword signal, not a grade -- see the solutions file for what actually matters)
  you didn't mention: TTL vs. invalidate-on-write, cache invalidation, shrinking vs. closing the staleness window

Follow-up: Does this actually fix the underlying bug, or just shrink the window? What's the real fix?

Question 2/5  --  Chapter 5, junior, Latency triage
You're asked to name the four stages that make up a single request's end-to-end latency.
(no interactive input available in this run -- using a placeholder answer so the rest of this cell logic still executes for real)

(elapsed: 0s -- soft timer, nothing was cut off)
(rough keyword signal, not a grade -- see the s

## Session summary

No model answers are shown here or anywhere else in this notebook — review your answers
against `solutions/question_bank_answers.json` on your own, after the session.

In [7]:
print(f"Session complete. {len(asked)} question(s) asked:")
for qid in asked:
    print(f"  - {qid}")
print(f"\nReview your answers against: solutions/question_bank_answers.json")
print(f"(look up each id above in that file for a full written model answer)")


Session complete. 5 question(s) asked:
  - reliability-005
  - latency-004
  - reliability-006
  - cost-003
  - tools-001

Review your answers against: solutions/question_bank_answers.json
(look up each id above in that file for a full written model answer)
